In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

In [2]:
df = pd.read_csv('C:\\Users\\MONJYEEMAN\\Documents\\Programming\\ML Project\\MechAssist\\data\\module3\\ai4i2020.csv')
df.shape

(10000, 14)

In [3]:
df.columns

Index(['UDI', 'Product ID', 'Type', 'Air temperature [K]',
       'Process temperature [K]', 'Rotational speed [rpm]', 'Torque [Nm]',
       'Tool wear [min]', 'Machine failure', 'TWF', 'HDF', 'PWF', 'OSF',
       'RNF'],
      dtype='str')

In [4]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 10000 entries, 0 to 9999
Data columns (total 14 columns):
 #   Column                   Non-Null Count  Dtype  
---  ------                   --------------  -----  
 0   UDI                      10000 non-null  int64  
 1   Product ID               10000 non-null  str    
 2   Type                     10000 non-null  str    
 3   Air temperature [K]      10000 non-null  float64
 4   Process temperature [K]  10000 non-null  float64
 5   Rotational speed [rpm]   10000 non-null  int64  
 6   Torque [Nm]              10000 non-null  float64
 7   Tool wear [min]          10000 non-null  int64  
 8   Machine failure          10000 non-null  int64  
 9   TWF                      10000 non-null  int64  
 10  HDF                      10000 non-null  int64  
 11  PWF                      10000 non-null  int64  
 12  OSF                      10000 non-null  int64  
 13  RNF                      10000 non-null  int64  
dtypes: float64(3), int64(9), str(2)
me

In [5]:
df['Machine failure'].value_counts()

Machine failure
0    9661
1     339
Name: count, dtype: int64

In [6]:
df['Type'].value_counts()

Type
L    6000
M    2997
H    1003
Name: count, dtype: int64

In [7]:
df[['Air temperature [K]', 'Process temperature [K]', 'Rotational speed [rpm]', 'Torque [Nm]', 'Tool wear [min]']].describe()

,Air temperature [K],Process temperature [K],Rotational speed [rpm],Torque [Nm],Tool wear [min]
count,10000.000000,10000.000000,10000.000000,10000.000000,10000.000000
mean,300.004930,310.005560,1538.776100,39.986910,107.951000
std,2.000259,1.483734,179.284096,9.968934,63.654147
min,295.300000,305.700000,1168.000000,3.800000,0.000000
25%,298.300000,308.800000,1423.000000,33.200000,53.000000
50%,300.100000,310.100000,1503.000000,40.100000,108.000000
75%,301.500000,311.100000,1612.000000,46.800000,162.000000
max,304.500000,313.800000,2886.000000,76.600000,253.000000


In [8]:
# Drop columns we don't need
df = df.drop(columns=['UDI', 'Product ID', 'RNF'])

# Encode Type: L=0, M=1, H=2
df['Type'] = df['Type'].map({'L': 0, 'M': 1, 'H': 2})

df.head()

,Type,Air temperature [K],Process temperature [K],Rotational speed [rpm],Torque [Nm],Tool wear [min],Machine failure,TWF,HDF,PWF,OSF
0,1,298.1,308.6,1551,42.8,0,0,0,0,0,0
1,0,298.2,308.7,1408,46.3,3,0,0,0,0,0
2,0,298.1,308.5,1498,49.4,5,0,0,0,0,0
3,0,298.2,308.6,1433,39.5,7,0,0,0,0,0
4,0,298.2,308.7,1408,40.0,9,0,0,0,0,0


In [9]:
# Taylor's tool life proxy: cutting power = Torque × angular velocity
# Angular velocity (rad/s) = rpm × 2π / 60
df['cutting_power'] = df['Torque [Nm]'] * (df['Rotational speed [rpm]'] * 2 * np.pi / 60)

# Merchant's chip thickness ratio proxy: process temp delta / air temp
df['temp_delta'] = df['Process temperature [K]'] - df['Air temperature [K]']

df[['cutting_power', 'temp_delta']].describe()

,cutting_power,temp_delta
count,10000.000000,10000.000000
mean,6279.744953,10.000630
std,1067.418295,1.001094
min,1148.440610,7.600000
25%,5561.184484,9.300000
50%,6271.027344,9.800000
75%,7003.002724,11.000000
max,10469.923005,12.100000


In [10]:
from sklearn.ensemble import RandomForestRegressor, RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error, r2_score, classification_report

FEATURES = ['Type', 'Air temperature [K]', 'Process temperature [K]',
            'Rotational speed [rpm]', 'Torque [Nm]', 'TWF', 'HDF', 
            'PWF', 'OSF', 'cutting_power', 'temp_delta']

X = df[FEATURES]
y_reg = df['Tool wear [min]']
y_clf = df['Machine failure']

X_train, X_test, yr_train, yr_test, yc_train, yc_test = train_test_split(
    X, y_reg, y_clf, test_size=0.2, random_state=42
)

In [11]:
# ── Regressor ──────────────────────────────────────────────────────────────
rf_reg = RandomForestRegressor(n_estimators=100, random_state=42)
rf_reg.fit(X_train, yr_train)

yr_pred = rf_reg.predict(X_test)
print(f"MAE: {mean_absolute_error(yr_test, yr_pred):.2f} minutes")
print(f"R²: {r2_score(yr_test, yr_pred):.4f}")

MAE: 50.15 minutes
R²: 0.1214


In [12]:
df[FEATURES + ['Tool wear [min]']].corr()['Tool wear [min]'].sort_values(ascending=False)

Tool wear [min]            1.000000
OSF                        0.155894
TWF                        0.115792
Air temperature [K]        0.013853
Process temperature [K]    0.013488
Rotational speed [rpm]     0.000223
HDF                       -0.001287
Torque [Nm]               -0.003093
cutting_power             -0.003193
Type                      -0.007255
temp_delta                -0.007689
PWF                       -0.009334
Name: Tool wear [min], dtype: float64

In [13]:
# ── Classifier ─────────────────────────────────────────────────────────────
rf_clf = RandomForestClassifier(n_estimators=100, class_weight='balanced', random_state=42)
rf_clf.fit(X_train, yc_train)

yc_pred = rf_clf.predict(X_test)
print(classification_report(yc_test, yc_pred, target_names=['No Failure', 'Failure']))

              precision    recall  f1-score   support

  No Failure       1.00      1.00      1.00      1939
     Failure       1.00      0.97      0.98        61

    accuracy                           1.00      2000
   macro avg       1.00      0.98      0.99      2000
weighted avg       1.00      1.00      1.00      2000



In [14]:
import joblib
import os

os.makedirs('C:\\Users\\MONJYEEMAN\\Documents\\Programming\\ML Project\\MechAssist\\models', exist_ok=True)

joblib.dump(rf_clf, 'C:\\Users\\MONJYEEMAN\\Documents\\Programming\\ML Project\\MechAssist\\models\\rf_module3_clf.pkl')
print("Model saved")

Model saved


In [15]:
import sys
sys.path.append('..')
import importlib
import modules.module3_machining as m3
importlib.reload(m3)

results = m3.get_advisory(
    type_grade='M',
    air_temp=300.0,
    process_temp=310.0,
    rpm=1500,
    torque=40.0,
    v_base=150.0,
    feed_base=0.2,
    depth_base=2.0
)

for r in results:
    print(r)

{'Mode': 'Conservative', 'speed (m/min)': 120.0, 'feed (mm/rev)': 0.16, 'depth (mm)': 1.6, 'tool_life (min)': 7.72, 'Failure Risk': 'No', 'Confidence (%)': np.float64(100.0)}
{'Mode': 'Balanced', 'speed (m/min)': 150.0, 'feed (mm/rev)': 0.2, 'depth (mm)': 2.0, 'tool_life (min)': 3.16, 'Failure Risk': 'No', 'Confidence (%)': np.float64(100.0)}
{'Mode': 'Aggressive', 'speed (m/min)': 180.0, 'feed (mm/rev)': 0.24, 'depth (mm)': 2.4, 'tool_life (min)': 1.52, 'Failure Risk': 'No', 'Confidence (%)': np.float64(100.0)}
